In [ ]:
import logging
import os

import xarray as xr
import numpy as np
import pcraster as pcr
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import geopandas as gpd
import resevoir_functions as rf

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

In [2]:
def estimate_discharge_for_environmental_flow(self, channelStorage):
    # z-score for 90th percentile; sets the environmental flow threshold
    z_score = 1.2816

    # Long-term variance of discharge using an online Welford algorithm
    varDischarge = self.m2tDischarge / \
                   pcr.max(1.,
                   pcr.min(self.maxTimestepsToAvgDischargeLong, self.timestepsToAvgDischarge) - 1.)
    stdDischarge = pcr.max(varDischarge**0.5, 0.0)

    # Floor at 10% of avg discharge to prevent flip-flop near the threshold
    minDischargeForEnvironmentalFlow = pcr.max(0.0, self.avgDischarge - z_score * stdDischarge)
    factor = 0.10
    minDischargeForEnvironmentalFlow = pcr.max(factor * self.avgDischarge, minDischargeForEnvironmentalFlow)
    minDischargeForEnvironmentalFlow = pcr.max(0.0, minDischargeForEnvironmentalFlow)

    return minDischargeForEnvironmentalFlow

#This function is most likely redundant as we have the net env flow from the netCDF files.

In [3]:
#loading data
def load_data(file_path):
    try:
        data = xr.open_dataset(file_path)
        logger.info(f"Data loaded successfully from {file_path}")
        return data
    except Exception as e:
        logger.error(f"Error loading data from {file_path}: {e}")
        return None

file_path = os.path.join(os.getcwd(), 'Data', 'POINTDATA') + os.sep

latitude = 37.625072479248
longitude = -122.458351135254

# Monthly averages; daily resolution can be re-run for a select region
discharge = load_data(file_path + "sos_resout_final_monthAvg_output.nc").sel(lat=latitude, lon=longitude, method='nearest').to_dataframe().reset_index().drop(columns={'lon', 'lat'})
inflow = load_data(file_path + "soswaterInflow_annuaTot_output.nc").sel(lat=latitude, lon=longitude, method='nearest').to_dataframe().reset_index().drop(columns={'lon', 'lat'})
env_flow = load_data(file_path + 'soswater_env_flow_monthTot_output.nc').sel(lat=latitude, lon=longitude, method='nearest').to_dataframe().reset_index().drop(columns={'lon', 'lat'})
Storage_validation = load_data(file_path + 'sosStor_check_dailyTot_output.nc').sel(lat=latitude, lon=longitude, method='nearest').to_dataframe().reset_index().drop(columns={'lon', 'lat'})

# sosreduction_demand_*_output.nc files are corrupt: they contain storage values instead of
# actual demand. We reconstruct demand by summing the five sector-specific monthly withdrawal
# files that PCR-GLOBWB exports separately.
#
# Unit pipeline:
#   monthTot files → m depth (sum of daily m/day values over the month)
#   × cell_area_m2 → m³/month
#
# Cell area varies with latitude on a regular lat/lon grid (5 arcmin ≈ 0.0833°).
cell_area_m2 = (5/60 * np.pi/180)**2 * 6_371_000**2 * np.cos(np.radians(latitude))

# The five sectors that make up total water demand on the reservoir
withdrawal_files = {
    'domestic':     'domesticWaterWithdrawal_monthTot_output.nc',
    'industry':     'industryWaterWithdrawal_monthTot_output.nc',
    'irr_nonpaddy': 'irrNonPaddyWaterWithdrawal_monthTot_output.nc',
    'irr_paddy':    'irrPaddyWaterWithdrawal_monthTot_output.nc',
    'livestock':    'livestockWaterWithdrawal_monthTot_output.nc',
}

# Load each sector at the dam grid cell and rename its variable column to the sector name
demand_parts = []
for sector, fname in withdrawal_files.items():
    df_part = load_data(file_path + fname).sel(lat=latitude, lon=longitude, method='nearest').to_dataframe().reset_index().drop(columns=['lat', 'lon'])
    df_part = df_part.rename(columns={df_part.columns[1]: sector})
    demand_parts.append(df_part)

# Merge all sectors on time, then sum and convert to m³/month
demand_tot = demand_parts[0].copy()
for df_part in demand_parts[1:]:
    demand_tot = pd.merge(demand_tot, df_part, on='time')

demand_tot['demand_m3'] = demand_tot[list(withdrawal_files.keys())].sum(axis=1) * cell_area_m2
demand_tot = demand_tot[['time', 'demand_m3']]

# Calculate the number of days in each month and the monthly inflow based on the daily inflow
inflow['year'] = inflow['time'].dt.year

output = pd.merge(demand_tot, discharge, on='time')
output = pd.merge(output, Storage_validation, on='time')
output = pd.merge(output, env_flow, on='time')

output['year'] = output['time'].dt.year
output = pd.merge(output, inflow[['year', 'soswater_inflow']], on='year')

output['days_in_month'] = output['time'].dt.days_in_month
output['inflow_monthly'] = (output['soswater_inflow'] / 365) * output['days_in_month']
output['modelled_storage'] = 0.0
output['model_release'] = 0.0
output['flood'] = 0.0
output['conservation'] = 0.0
output['model_current_storage'] = 0.0
output['reduction_factor_model'] = 0.0
output['day'] = range(0, len(output))

df = output.copy()
cap_215 = 23.5 * 1e6
avg_inflow = inflow['soswater_inflow'].mean()

#import flood and conservation bounds
rf_dir = os.path.join('Data', 'POINTDATA', '10_param_RF_bounds_final')

bounds = {} #fractions of capacity
for fname in os.listdir(rf_dir):
    if fname.endswith('.nc'):
        ds = xr.open_dataset(os.path.join(rf_dir, fname))
        row = ds.sel(latitude=latitude, longitude=longitude, method='nearest')
        key = fname.replace('.nc', '')  # e.g. "2000-3-15"
        bounds[key] = (float(row['flood'])/100, float(row['conservation'])/100) 


2026-05-18 18:45:23,078 INFO Data loaded successfully from c:\Users\dlp96\Documents\GitHub\resevoir_study\Data\POINTDATA\sos_resout_final_monthAvg_output.nc
2026-05-18 18:45:23,087 INFO Data loaded successfully from c:\Users\dlp96\Documents\GitHub\resevoir_study\Data\POINTDATA\soswaterInflow_annuaTot_output.nc
2026-05-18 18:45:23,095 INFO Data loaded successfully from c:\Users\dlp96\Documents\GitHub\resevoir_study\Data\POINTDATA\soswater_env_flow_monthTot_output.nc
2026-05-18 18:45:23,190 INFO Data loaded successfully from c:\Users\dlp96\Documents\GitHub\resevoir_study\Data\POINTDATA\sosStor_check_dailyTot_output.nc
2026-05-18 18:45:23,281 INFO Data loaded successfully from c:\Users\dlp96\Documents\GitHub\resevoir_study\Data\POINTDATA\domesticWaterWithdrawal_monthTot_output.nc
2026-05-18 18:45:23,292 INFO Data loaded successfully from c:\Users\dlp96\Documents\GitHub\resevoir_study\Data\POINTDATA\industryWaterWithdrawal_monthTot_output.nc
2026-05-18 18:45:23,303 INFO Data loaded success

In [4]:
df

,time,demand_m3,sos_reservoir_outflow_end,sos_storage_check,soswater_env_flow,year,soswater_inflow,days_in_month,inflow_monthly,modelled_storage,model_release,flood,conservation,model_current_storage,reduction_factor_model,day
0,1979-01-31,5.595388e+05,4021.334229,23500000.0,1.442840,1979,41645380.0,31,3.537005e+06,0.0,0.0,0.0,0.0,0.0,0.0,0
1,1979-02-28,5.909822e+05,4336.654297,23500000.0,1.405397,1979,41645380.0,28,3.194714e+06,0.0,0.0,0.0,0.0,0.0,0.0,1
2,1979-03-31,5.984688e+05,4816.763672,23500000.0,1.728237,1979,41645380.0,31,3.537005e+06,0.0,0.0,0.0,0.0,0.0,0.0,2
3,1979-04-30,6.164364e+05,5186.385742,23500000.0,1.800828,1979,41645380.0,30,3.422908e+06,0.0,0.0,0.0,0.0,0.0,0.0,3
4,1979-05-31,6.373987e+05,4849.555176,23500000.0,1.927865,1979,41645380.0,31,3.537005e+06,0.0,0.0,0.0,0.0,0.0,0.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535,2023-08-31,2.137647e+06,0.000000,17700000.0,2.699958,2023,41039172.0,31,3.485519e+06,0.0,0.0,0.0,0.0,0.0,0.0,535
536,2023-09-30,2.078716e+06,0.000000,17700000.0,2.574645,2023,41039172.0,30,3.373083e+06,0.0,0.0,0.0,0.0,0.0,0.0,536
537,2023-10-31,1.879256e+06,0.000000,17700000.0,2.620943,2023,41039172.0,31,3.485519e+06,0.0,0.0,0.0,0.0,0.0,0.0,537
538,2023-11-30,1.752112e+06,238.140457,17700000.0,2.496387,2023,41039172.0,30,3.373083e+06,0.0,0.0,0.0,0.0,0.0,0.0,538


In [ ]:
df_simulation = df.copy()
for t, row in df_simulation.iterrows():

    storage_capacity = df_simulation['sos_storage_check'].max() #probably not the way to go, however when i built a function out of this so that every dam can be inserted based on their id i can use this:
    #storage_capacity = input_data[input_data['hydrolakes'] == dam_id]['cap'].values[0]

    key = f"2000-{row['time'].month}-{row['time'].day}"
    flood_frac, conservation_frac = bounds[key] #get the flood and conservation bounds

    max_storage = flood_frac * cap_215
    min_storage = conservation_frac * cap_215

    if t == 0: #get initial conditions
        current_storage = storage_capacity / 2 #df['sos_storage_check'][0] this is a placeholder, we want the pcr-globw storage value but it appears larger than the max
    else:
        # carry forward storage from the previous timestep
        current_storage = df_simulation.loc[t - 1, 'modelled_storage']

    logger.debug('flood and conservation values %s %s', max_storage, min_storage)

    #start calculating
    reduction_factor = rf.reduction_factor(current_storage=current_storage, min_storage=min_storage, max_storage=max_storage)
    release = rf.starfit_release(current_storage=current_storage, storage_capacity=storage_capacity, max_storage=max_storage, min_storage=min_storage, avg_outflow=row['sos_reservoir_outflow_end'], env_flow=row['soswater_env_flow'], demand=row['demand_m3'], use='irrigation')
    df_simulation.loc[t, 'model_release'] = release

    new_storage = rf.new_storage(release=release, current_storage=current_storage, inflow=row['inflow_monthly'])
    df_simulation.loc[t, 'modelled_storage'] = new_storage


In [9]:
df_simulation

,time,demand_m3,sos_reservoir_outflow_end,sos_storage_check,soswater_env_flow,year,soswater_inflow,days_in_month,inflow_monthly,modelled_storage,model_release,flood,conservation,model_current_storage,reduction_factor_model,day
0,1979-01-31,5.595388e+05,4021.334229,23500000.0,1.442840,1979,41645380.0,31,3.537005e+06,1.528700e+07,1.44284,0.0,0.0,0.0,0.0,0
1,1979-02-28,5.909822e+05,4336.654297,23500000.0,1.405397,1979,41645380.0,28,3.194714e+06,0.000000e+00,0.00000,0.0,0.0,0.0,0.0,1
2,1979-03-31,5.984688e+05,4816.763672,23500000.0,1.728237,1979,41645380.0,31,3.537005e+06,0.000000e+00,0.00000,0.0,0.0,0.0,0.0,2
3,1979-04-30,6.164364e+05,5186.385742,23500000.0,1.800828,1979,41645380.0,30,3.422908e+06,0.000000e+00,0.00000,0.0,0.0,0.0,0.0,3
4,1979-05-31,6.373987e+05,4849.555176,23500000.0,1.927865,1979,41645380.0,31,3.537005e+06,0.000000e+00,0.00000,0.0,0.0,0.0,0.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535,2023-08-31,2.137647e+06,0.000000,17700000.0,2.699958,2023,41039172.0,31,3.485519e+06,0.000000e+00,0.00000,0.0,0.0,0.0,0.0,535
536,2023-09-30,2.078716e+06,0.000000,17700000.0,2.574645,2023,41039172.0,30,3.373083e+06,0.000000e+00,0.00000,0.0,0.0,0.0,0.0,536
537,2023-10-31,1.879256e+06,0.000000,17700000.0,2.620943,2023,41039172.0,31,3.485519e+06,0.000000e+00,0.00000,0.0,0.0,0.0,0.0,537
538,2023-11-30,1.752112e+06,238.140457,17700000.0,2.496387,2023,41039172.0,30,3.373083e+06,0.000000e+00,0.00000,0.0,0.0,0.0,0.0,538


In [ ]:
# Normalise storage and release by capacity, then melt to long format for seaborn
plot_df = df_simulation[['time', 'sos_storage_check', 'modelled_storage', 'model_release', 'inflow_monthly']].copy()
plot_df['PCR storage']      = plot_df['sos_storage_check']  / cap_215
plot_df['modelled storage'] = plot_df['modelled_storage']   / cap_215
plot_df['modelled release'] = plot_df['model_release']      / cap_215
plot_df['inflow']           = plot_df['inflow_monthly']     / cap_215

plot_long = plot_df.melt(
    id_vars='time',
    value_vars=['PCR storage', 'modelled storage', 'modelled release', 'inflow'],
    var_name='series',
    value_name='fraction of capacity'
)

fig, ax = plt.subplots(figsize=(14, 5))
sns.lineplot(data=plot_long, x='time', y='fraction of capacity', hue='series', ax=ax)
ax.axhline(y=1.0, color='black', linestyle='-', linewidth=0.8, label='full capacity')
ax.set_title('Reservoir storage — PCR-GLOBWB vs modelled')
ax.set_xlabel('Time')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
